# 第9回　正則化
***
> **前提**: 第3回・第7回の回帰分析を発展させ，正則化とロジスティック回帰を学びます。

## 目次
1. データの読み込み
2. Ridge / Lasso 回帰
3. ロジスティック回帰の正則化
4. 正則化強度の比較

---

## この回で学ぶこと

### 過学習（Overfitting）とは何か

モデルが訓練データに**過度に適合**してしまい，未知のデータ（テストデータ）では精度が落ちる現象を「過学習」という。特に特徴量が多い場合（変数の数 > サンプル数），線形モデルでも簡単に過学習が起きる。

```
【過学習の兆候】
訓練 R² = 0.98  テスト R² = 0.42  ← 大きく乖離 → 過学習

【理想的な状態】
訓練 R² = 0.82  テスト R² = 0.79  ← ほぼ同じ → 汎化できている
```

### 正則化の直感的な理解

正則化とは「モデルの係数が大きくなりすぎることにペナルティを課す」仕組みだ。

通常の線形回帰の損失関数：**誤差の二乗和（RSS）**を最小化

正則化では損失関数に「係数の大きさ」を加える：
- **Ridge（L2正則化）**: RSS + α × Σ(係数²)
- **Lasso（L1正則化）**: RSS + α × Σ|係数|

αが大きいほど正則化が強く，係数は0に近づく（モデルが単純になる）。

### Ridge vs Lasso の本質的な違い

| 比較軸 | Ridge（L2） | Lasso（L1） |
|---|---|---|
| 係数の扱い | すべての係数を小さくする（0にはしない） | 不要な特徴量の係数を**完全に0にする** |
| 特徴量選択 | できない | **できる**（スパースモデル） |
| 向いている場面 | 全特徴量が多少は予測に寄与する場合 | 重要な特徴量が少数に絞られる場合 |
| 計算 | 解析解あり（安定） | 反復計算が必要 |

> **卒業研究での活用**: 説明変数が多い（遺伝子発現データ，アンケート，センサーデータなど）場合，Lasso で重要変数を自動選択する手法は非常によく使われる。

### ロジスティック回帰の正則化パラメータ C

scikit-learn の `LogisticRegression` では正則化強度を **C** で指定する。Cは `alpha` の逆数だ：
- **C が小さい** → 正則化が強い → 係数が小さい → シンプルなモデル
- **C が大きい** → 正則化が弱い → 係数が自由 → 複雑なモデル（過学習しやすい）

混乱しやすいので注意：Ridge の `alpha` と `LogisticRegression` の `C` は**逆の関係**だ。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATA_BASE = "https://raw.githubusercontent.com/ShotaYmzk/AI-kadai/main/data"

def load_student_csv(filename: str) -> pd.DataFrame:
    return pd.read_csv(f"{DATA_BASE}/student/{filename}", sep=";")


## 問題1　ベースラインの線形回帰
***

### データについて

UCI の学生成績データには，ポルトガルの数学と国語の成績が含まれている。
- `G1`: 1学期の成績（0〜20点）
- `G2`: 2学期の成績（0〜20点）
- `G3`: 最終成績（0〜20点）← 予測したい目的変数

今回は最もシンプルな設定として `G1` → `G3` を予測する。

### なぜ標準化が必要か（回帰の場合）

線形回帰自体はスケールに影響されないが，正則化（Ridge/Lasso）は**係数の大きさにペナルティ**をかけるため，スケールが異なると不公平なペナルティになる。問題2以降で Ridge/Lasso を使うため，ここで標準化しておく。

### 評価指標の意味

- **MSE（Mean Squared Error）**: 予測誤差の二乗の平均。値が小さいほど良い。単位は（得点²）なので直感的に分かりにくい
- **RMSE（Root MSE）**: MSEの平方根。元の単位（点）で誤差を表せる
- **R²（決定係数）**: 0〜1の範囲で，1が完璧な予測。「目的変数の分散のうちモデルが説明できる割合」

### 課題

UCI の学生成績データ `student-mat.csv` を URL から読み込み，説明変数を `G1`，目的変数を `G3` として train/test（80:20）に分割してください。

説明変数を `StandardScaler` で標準化したうえで，`LinearRegression` の**テスト MSE** と **R2 スコア**を出力してください。

> **考えてみよう**: G1 だけで G3 をどの程度予測できるか？R² が高ければ「1学期の成績が最終成績を概ね決定する」ことを示す。

#### Hints
- 評価には `mean_squared_error` と `r2_score` 関数を使う。引数の順番は `(正解値, 予測値)` だ
- 標準化は「訓練データで `fit` し、その統計量で訓練・テスト両方を `transform`」する。テストデータに `fit` するとデータリークになる
- `X` は1列だが `scaler` は2次元配列を期待するため、`X.values.reshape(-1, 1)` などで形状を合わせる必要がある

In [ ]:
# データ読み込み
# ここにあなたのコードを書いてください


# 標準化と線形回帰
# ここにあなたのコードを書いてください


## 問題2　Ridge と Lasso の比較
***

### alpha の意味と選び方

今回は `Ridge(alpha=1.0)` と `Lasso(alpha=0.1)` を使う。alpha の値はハイパーパラメータ（事前に設定する値）であり，通常は交差検証で最適値を探す（第13回で扱う）。

今回 Lasso の alpha を Ridge より小さくしている理由：Lasso は正則化が強すぎると係数が全部0になってしまうため，弱めの正則化から始めることが多い。

### 特徴量が1つの場合の注意

今回は `G1` のみを説明変数にしているため，Ridge と Lasso の差は小さく，LinearRegression とほぼ同じ結果になる可能性が高い。正則化の効果は**説明変数が多い（多変量）**場合に顕著になる。

> **発展**: `G1`, `G2`, `studytime`, `failures`, `absences` など複数の特徴量で試すと，Lasso が不要な特徴量の係数を0にする様子が観察できる。

### 課題

同じデータに対して `Ridge(alpha=1.0)` と `Lasso(alpha=0.1)` を学習し，それぞれのテスト R2 スコアを出力してください。

LinearRegression・Ridge・Lasso の R² を並べて比較し，どれが最も良いか確認してください。

#### Hints
- `Ridge` / `Lasso` も `LinearRegression` と同じく `fit` → `predict` → 評価の流れで使える
- **発展**: 学習後にモデルの `.coef_` 属性を確認すると、正則化による係数の変化が観察できる


In [ ]:
# Ridge と Lasso
# ここにあなたのコードを書いてください


## 問題3　ロジスティック回帰による2値分類
***

### 回帰から分類へ

「点数を予測する（回帰）」から「合格/不合格を予測する（分類）」に問題を変換する。機械学習の実務では，最終的な判断が「Yes/No」「合格/不合格」「異常/正常」であることが多く，この変換はよく行われる。

### ロジスティック回帰とは

名前に「回帰」が入っているが，実際は**分類モデル**だ。内部では：

1. 線形結合を計算: z = w₁x₁ + w₂x₂ + ... + b
2. シグモイド関数で確率に変換: P(y=1) = 1 / (1 + e^(-z))
3. P > 0.5 なら 1（合格），P ≤ 0.5 なら 0（不合格）と予測

シグモイド関数は出力を必ず [0, 1] の範囲に収めるため，確率として解釈できる。

### 説明変数の選択について

今回使う変数の意味：
- `G1`, `G2`: 1〜2学期の成績（最も予測力が高いはず）
- `studytime`: 週の勉強時間（1=2時間未満 〜 4=10時間以上）
- `failures`: 過去の留年回数

> **考えてみよう**: なぜ `G1`, `G2` だけでなく `studytime` や `failures` も加えるのか？単純に成績だけでなく「学習習慣」や「過去の失敗」が最終成績の予測に寄与するかを確かめたいからだ。

### 課題

`student-mat.csv` の `G3`（0〜20点）を2値化し，`G3 >= 10` を 1，それ以外を 0 とした目的変数を作成してください。説明変数は `G1`, `G2`, `studytime`, `failures` とします。

`LogisticRegression(C=1.0, max_iter=1000)` を学習し，テストデータの正解率を出力してください。

#### Hints
- `G3 >= 10` という比較式は True/False の Series を返す。それを整数（0/1）に変換するメソッドがある
- `LogisticRegression` のデフォルト `max_iter=100` では反復が収束しない場合があるため、大きな値を指定する
- 標準化は必須ではないが、適用すると収束が安定する

In [ ]:
# 2値分類データの作成とロジスティック回帰
# ここにあなたのコードを書いてください


## 問題4　正則化強度の比較実験
***

### ハイパーパラメータ探索の考え方

C の値（正則化強度）は事前に「最適値」がわからない。そのため，複数の値を試して**テストデータでの性能を比較**する。これをハイパーパラメータ探索（Grid Search）という（第13回で自動化する方法を学ぶ）。

### なぜ対数スケールで比較するか

C = [0.01, 0.1, 1, 10, 100] は1000倍の範囲を探索している。線形スケールでは 0.01, 0.1, 1 が密集して見えてしまう。対数スケールにすることで各値を等間隔に表示でき，変化の傾向が見やすくなる。

### グラフから何を読み取るか

このグラフで典型的に見られるパターン：
- **C が小さすぎる（左端）**: 正則化が強すぎて underfitting（過少適合）→ 訓練もテストも低精度
- **C が大きすぎる（右端）**: 正則化が弱すぎて overfitting（過学習）→ 訓練は高精度だがテストは低下
- **最適な C**: テスト精度が最大になるあたり

> **卒業研究での応用**: このようなグラフを描いて最適なハイパーパラメータを決定するプロセスは，あらゆる機械学習研究で必要になる。論文にはこのような「ハイパーパラメータ感度分析」が含まれることが多い。

### 課題

`LogisticRegression` の `C` を `[0.01, 0.1, 1, 10, 100]` と変化させ，テスト正解率をグラフで比較してください。

横軸: C（対数スケール），縦軸: 正解率 の折れ線グラフを描画し，**最も正解率が高い C の値**を出力してください。

#### Hints
- C の値をリストにして `for` ループで各 C の正解率を計算し、リストに蓄積してからグラフを描く
- 横軸に C の値を並べる際、`plt.xscale("log")` で対数スケールにすると 0.01〜100 の範囲が見やすくなる
- **発展**: 訓練スコアと検証スコアを両方プロットすると、C が大きくなるにつれて過学習の傾向が視覚的に確認できる

In [ ]:
# C を変えた正則化強度の比較
# ここにあなたのコードを書いてください
